# Tokenizer va Dataset builder




| A - Tokenizer | B - Dataset builder |
|---|---|
| Nega subword | Split before encoding |
| BPE merge rules | uint16 binary |
| Vocabulary size | Sequence packing |
| Special tokens | Input / target shift |
| `train_tokenizer.py` | `meta.json` |
| Round-trip test | To'rtta tekshiruv |




In [ ]:
# Colab uchun. Lokalda `pip install -r requirements.txt` allaqachon bajarilgan.
# !pip install -q "tokenizers>=0.20,<1.0" numpy tqdm

In [1]:
import json
import random
import time
from collections import Counter
from pathlib import Path

import numpy as np
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers

try:
    from tqdm.auto import tqdm
except ImportError:                       # tqdm bo'lmasa ham ishlayveradi
    def tqdm(x, **kwargs):
        return x

import tokenizers
print("numpy      :", np.__version__)
print("tokenizers :", tokenizers.__version__)

numpy      : 2.4.6
tokenizers : 0.23.1


In [2]:
from pathlib import Path

print("Joriy ishchi papka:")
print(Path.cwd())

print("\nJoriy papkada data bormi?")
print((Path.cwd() / "data").exists())

print("\nBir daraja yuqorida data bormi?")
print((Path.cwd().parent / "data").exists())

Joriy ishchi papka:
c:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\5 dars\my_LLM

Joriy papkada data bormi?
True

Bir daraja yuqorida data bormi?
True


In [3]:
from pathlib import Path

# Joriy yoki yuqoridagi data papkasini qidiramiz
cwd = Path.cwd()

if (cwd / "data").is_dir():
    DATA_PATH = cwd / "data"

elif (cwd.parent / "data").is_dir():
    DATA_PATH = cwd.parent / "data"

else:
    raise FileNotFoundError(
        "data papkasi topilmadi!"
    )

print("DATA PATH:", DATA_PATH.resolve())

print("\nDATA PAPKASIDAGI FAYLLAR:")

for file in DATA_PATH.rglob("*"):
    if file.is_file():
        print(file.relative_to(DATA_PATH))

DATA PATH: C:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\5 dars\my_LLM\data

DATA PAPKASIDAGI FAYLLAR:
cleaned\simplestories_cleaned.jsonl
filtered\simplestories_filtered.jsonl
final\run_config.json
final\simplestories_final.jsonl
final\stats_report.md
raw\simplestories_train.jsonl


In [4]:
from pathlib import Path

print("Python ishlayotgan papka:")
print(Path.cwd())

print("\nPROJECT_ROOT:")
print(Path("..").resolve())

Python ishlayotgan papka:
c:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\5 dars\my_LLM

PROJECT_ROOT:
C:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\5 dars


## 0. Konfiguratsiya



In [6]:
# Bu notebook my_LLM/tokenizer/ ichida turibdi -> loyiha ildizi bir daraja yuqorida
from pathlib import Path

PROJECT_ROOT = Path.cwd()

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())

FINAL_FILE    = PROJECT_ROOT / "data" / "final" / "simplestories_final.jsonl"
TOKENIZER_DIR = PROJECT_ROOT / "tokenizer"
DATASET_DIR   = PROJECT_ROOT / "dataset"

TOKENIZER_FILE = TOKENIZER_DIR / "tokenizer.json"
TRAIN_BIN      = DATASET_DIR / "train.bin"
VAL_BIN        = DATASET_DIR / "val.bin"
META_FILE      = DATASET_DIR / "meta.json"

for d in (TOKENIZER_DIR, DATASET_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---------- Tokenizer ----------
VOCAB_SIZE     = 8000          # lug'at hajmi = embedding jadvalidagi qatorlar soni
MIN_FREQUENCY  = 2             # 1 marta uchragan juftlik merge bo'lmaydi
SPECIAL_TOKENS = ["<eos>"]     # nega faqat bittasi - 8-bo'limda

# BPE statistikasi tez to'yinadi: 200k hikoya 2M hikoya bilan deyarli bir xil
# merge'lar beradi, lekin darsda 10 daqiqa emas, ~1 daqiqa oladi.
TOKENIZER_SAMPLE_DOCS = 200_000

# ---------- Dataset ----------
BLOCK_SIZE = 256               # context length: model bir qadamda ko'radigan token soni
VAL_RATIO  = 0.01              # hujjatlarning 1% i validation'ga
SEED       = 1337

# Darsda tez sinash uchun: MAX_DOCS = 50_000. Haqiqiy build uchun None.
MAX_DOCS = None

random.seed(SEED)
np.random.seed(SEED)

print("Final corpus :", FINAL_FILE.resolve())
print("Tokenizer    :", TOKENIZER_FILE.resolve())
print("Dataset      :", DATASET_DIR.resolve())

PROJECT_ROOT: C:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\5 dars\my_LLM
Final corpus : C:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\5 dars\my_LLM\data\final\simplestories_final.jsonl
Tokenizer    : C:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\5 dars\my_LLM\tokenizer\tokenizer.json
Dataset      : C:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\5 dars\my_LLM\dataset


In [7]:
FINAL_FILE

WindowsPath('c:/Users/xolmu/OneDrive/Desktop/Darslar oyma oy/LLM 2/5 dars/my_LLM/data/final/simplestories_final.jsonl')

## 1. Datani yuklaymiz



In [8]:
def load_jsonl(path, max_docs=None):
    """JSONL fayldan matnlar ro'yxatini o'qiydi."""
    texts = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_docs is not None and i >= max_docs:
                break
            texts.append(json.loads(line)["text"])
    return texts


if not FINAL_FILE.exists():
    raise FileNotFoundError(
        str(FINAL_FILE) + " topilmadi.\n"
        "Avval data/experiment.ipynb ni to'liq ishga tushiring - u shu faylni yaratadi."
    )

docs = load_jsonl(FINAL_FILE, max_docs=MAX_DOCS)

total_chars = sum(len(t) for t in docs)
total_words = sum(len(t.split()) for t in docs)

print("Hujjatlar    : {:,}".format(len(docs)))
print("Belgilar     : {:,}".format(total_chars))
print("So'zlar      : {:,}".format(total_words))
print("O'rtacha     : {:.0f} belgi / hujjat".format(total_chars / len(docs)))
print()
print("Namuna:")
print(docs[0][:300])

Hujjatlar    : 10,000
Belgilar     : 12,356,186
So'zlar      : 2,244,570
O'rtacha     : 1236 belgi / hujjat

Namuna:
Eagerly, a girl named Kim went hiking with her friends. They found a secret passageway under a big rock. Curious, they crawled through it and entered a magical world where all the animals talked. They learned that the animals had their own struggles but always found a way to be happy. 

Inspired, th


---

# A. TOKENIZER



## 3. Uchta yo'l: character / word / subword

| | Character | Word | **Subword (BPE)** |
|---|---|---|---|
| Lug'at hajmi | ~100 | 100k+ | 8k - 50k (biz tanlaymiz) |
| Ketma-ketlik uzunligi | juda uzun | qisqa | o'rtacha |
| OOV (noma'lum so'z) | yo'q | **bor** -> `<unk>` | yo'q |
| Morfologiya | yo'qoladi | yo'qoladi | saqlanadi (`play` + `ing`) |
| Embedding narxi | arzon | qimmat | o'rtacha |

Jadvalga ishonmang - o'z korpusimizda o'lchaymiz.

In [9]:
sample = docs[:20_000]                      # o'lchov uchun namuna
sample_chars = sum(len(t) for t in sample)

# --- character-level ---
char_vocab = set()
for t in sample:
    char_vocab.update(t)
char_tokens = sample_chars                  # har bir belgi = 1 token

# --- word-level ---
word_counter = Counter()
for t in sample:
    word_counter.update(t.split())
word_tokens = sum(word_counter.values())

header = "{:<14}{:>12}{:>16}{:>16}".format("Usul", "Lugat", "Tokenlar", "Token/belgi")
print(header)
print("-" * len(header))
print("{:<14}{:>12,}{:>16,}{:>16.3f}".format(
    "character", len(char_vocab), char_tokens, char_tokens / sample_chars))
print("{:<14}{:>12,}{:>16,}{:>16.3f}".format(
    "word", len(word_counter), word_tokens, word_tokens / sample_chars))
print()

# Word-level lug'atining uzun dumi
singletons = sum(1 for c in word_counter.values() if c == 1)
print("Faqat 1 marta uchragan so'zlar: {:,} / {:,}  ({:.1%})".format(
    singletons, len(word_counter), singletons / len(word_counter)))
print()
print("Eng kam uchraydigan 10 ta so'z:", [w for w, _ in word_counter.most_common()[-10:]])

Usul                 Lugat        Tokenlar     Token/belgi
----------------------------------------------------------
character               77      12,356,186           1.000
word                35,182       2,244,570           0.182

Faqat 1 marta uchragan so'zlar: 12,897 / 35,182  (36.7%)

Eng kam uchraydigan 10 ta so'z: ['calmer', 'skull.', 'Gathering,', 'react,', 'temporary', 'dissolve,', '"Balloon', 'mess?"', '"Stick', 'uniqueness!']


### Nega subword

Yuqoridagi raqamlar ikkita chekka holatni ko'rsatadi.

**Character-level.** Lug'at kichkina, lekin ketma-ketlik ~4-5 barobar uzun.
Attention narxi $O(n^2)$ - ketma-ketlikni 4 barobar uzaytirish hisobni
**16 barobar** qimmatlashtiradi. Ustiga-ustak model `the` so'zini har safar
uchta belgidan qaytadan yig'ib olishi kerak.

**Word-level.** Qisqa, lekin lug'at hech qachon yopilmaydi. Yuqorida ko'rdik:
lug'atning katta qismi bir martalik so'zlar - ular embedding'da hech narsa
o'rganmaydi, faqat joy egallaydi. Yangi so'z kelsa `<unk>` va ma'lumot butunlay
yo'qoladi. `<unk>` ni model generatsiya qilsa - uni matnga qaytarib ham
bo'lmaydi.

**Subword** - o'rta yo'l. Bitta jumla:

> Tez-tez uchraydigan narsa bitta token bo'lsin, kam uchraydigani bo'laklarga bo'linsin.

```
" the"        -> [" the"]                # juda tez-tez -> bitta token
" tokenizer"  -> [" token", "izer"]      # kamroq -> ikkiga
" Qutbiddin"  -> [" Qut", "bid", "din"]  # deyarli uchramaydi -> bo'laklarga
```

Natija: lug'at **cheklangan** (biz tanlaymiz), OOV **umuman yo'q** (eng yomon
holatda baytlargacha tushadi), morfologiya saqlanadi - `play`, `playing`,
`player` bir xil `play` bo'lagini bo'lishadi va shu bo'lakning embedding'i
uchalasidan ham o'rganadi.

## 4. BPE: merge rules

**Byte Pair Encoding** - 1994 yilda siqish algoritmi sifatida o'ylab topilgan,
2015 yilda NLP ga olib kirilgan. Algoritm ajablanarli darajada sodda.

**O'rgatish (training):**

```
1. Matnni eng mayda birliklarga bo'l (belgilar / baytlar)
2. Takrorla, lug'at hajmi V ga yetguncha:
     a. barcha qo'shni juftliklar sonini sana
     b. eng ko'p uchraganini top
     c. uni BITTA yangi token qilib birlashtir
     d. bu birlashtirishni MERGE RULE sifatida ro'yxatga yoz
```

Har qadamda tanlov - shunchaki argmax:

$$(a, b)^{*} = \arg\max_{(a,b)} \; \text{count}(a, b)$$

$$V_{\text{yakuniy}} = V_{\text{boshlangich}} + (\text{merge lar soni}) + (\text{special token lar})$$

**Eng muhim nuqta:** natija - bu lug'at emas, balki **tartiblangan qoidalar
ro'yxati**. Yangi matnni kodlashda merge'lar aynan o'sha tartibda qo'llanadi.
Tartibni buzsangiz, boshqa tokenlar chiqadi.

Klassik misol bilan ko'ramiz.

In [12]:
# --- O'yinchoq BPE. Kutubxonasiz, 30 qator. ---

def apply_merge(word, pair):
    """word - bo'laklar tuple'i. pair - birlashtiriladigan juftlik."""
    out, i = [], 0
    while i < len(word):
        if i < len(word) - 1 and word[i] == pair[0] and word[i + 1] == pair[1]:
            out.append(word[i] + word[i + 1])      # ikkitasini bittaga qo'shdik
            i += 2
        else:
            out.append(word[i])
            i += 1
    return tuple(out)


def toy_bpe_train(text, num_merges, verbose=True):
    """Klassik so'z-darajasidagi BPE. '_' - so'z oxiri belgisi."""
    freqs = Counter(text.split())
    # har bir so'z: belgilar tuple'i + so'z oxiri markeri
    words = {tuple(list(w) + ["_"]): c for w, c in freqs.items()}

    merges = []
    for step in range(num_merges):
        # a. qo'shni juftliklarni sanaymiz (so'z chastotasi bilan vaznlab)
        pairs = Counter()
        for word, freq in words.items():
            for pair in zip(word, word[1:]):
                pairs[pair] += freq
        if not pairs:
            break

        # b. argmax
        best, count = pairs.most_common(1)[0]
        if count < 2:                              # min_frequency
            break

        # c + d. birlashtiramiz va qoidani yozib qo'yamiz
        words = {apply_merge(w, best): c for w, c in words.items()}
        merges.append(best)

        if verbose:
            print("merge {:2d}: {!r} + {!r}  ->  {!r}   (uchradi: {})".format(
                step + 1, best[0], best[1], best[0] + best[1], count))

    return merges, words


toy_text = ("play play play play play player player playing playing playing "
            "playing playing playing replay replay replay")

print("Matn:", toy_text)
print()
toy_merges, toy_words = toy_bpe_train(toy_text, num_merges=10)

Matn: play play play play play player player playing playing playing playing playing playing replay replay replay

merge  1: 'p' + 'l'  ->  'pl'   (uchradi: 16)
merge  2: 'pl' + 'a'  ->  'pla'   (uchradi: 16)
merge  3: 'pla' + 'y'  ->  'play'   (uchradi: 16)
merge  4: 'play' + '_'  ->  'play_'   (uchradi: 8)
merge  5: 'play' + 'i'  ->  'playi'   (uchradi: 6)
merge  6: 'playi' + 'n'  ->  'playin'   (uchradi: 6)
merge  7: 'playin' + 'g'  ->  'playing'   (uchradi: 6)
merge  8: 'playing' + '_'  ->  'playing_'   (uchradi: 6)
merge  9: 'r' + 'e'  ->  're'   (uchradi: 3)
merge 10: 're' + 'play_'  ->  'replay_'   (uchradi: 3)


In [13]:
print("O'rgatilgandan keyin so'zlar qanday bo'lakka bo'lingan:\n")
for word, freq in sorted(toy_words.items(), key=lambda kv: -kv[1]):
    print("  {:>3} x  {}".format(freq, list(word)))

print("\nMerge rules (TARTIB muhim):")
for i, (a, b) in enumerate(toy_merges, 1):
    print("  {:2d}. {!r} + {!r}".format(i, a, b))

O'rgatilgandan keyin so'zlar qanday bo'lakka bo'lingan:

    6 x  ['playing_']
    5 x  ['play_']
    3 x  ['replay_']
    2 x  ['play', 'e', 'r', '_']

Merge rules (TARTIB muhim):
   1. 'p' + 'l'
   2. 'pl' + 'a'
   3. 'pla' + 'y'
   4. 'play' + '_'
   5. 'play' + 'i'
   6. 'playi' + 'n'
   7. 'playin' + 'g'
   8. 'playing' + '_'
   9. 'r' + 'e'
  10. 're' + 'play_'


### Qoidalarni yangi so'zga qo'llash

Diqqat qiling: pastdagi funksiyada hech qanday "lug'atdan qidirish" yo'q.
Biz shunchaki **o'sha tartibda** qoidalarni ketma-ket qo'llaymiz. Shuning uchun
BPE hech qachon `<unk>` chiqarmaydi - eng yomon holatda so'z alohida
belgilarga bo'linib qoladi.

In [15]:
def toy_bpe_encode(word, merges):
    tokens = tuple(list(word) + ["_"])

    for pair in merges:  # Tartib o'rgatishdagidek
        tokens = apply_merge(tokens, pair)

    return list(tokens)


for w in ["player", "playing", "replay", "replaying", "rain"]:
    print("{:>10}  ->  {}".format(w, toy_bpe_encode(w, toy_merges)))


print()
print("'rain' o'quv matnida umuman yo'q edi, lekin <unk> chiqmadi -")
print("u belgilarga bo'linib qoldi. Mana shu BPE ning asosiy yutug'i.")

    player  ->  ['play', 'e', 'r', '_']
   playing  ->  ['playing_']
    replay  ->  ['replay_']
 replaying  ->  ['re', 'playing_']
      rain  ->  ['r', 'a', 'i', 'n', '_']

'rain' o'quv matnida umuman yo'q edi, lekin <unk> chiqmadi -
u belgilarga bo'linib qoldi. Mana shu BPE ning asosiy yutug'i.


## 5. Byte-level BPE

Yuqoridagi o'yinchoq versiya **belgilardan** boshladi. Haqiqiy tokenizer
(GPT-2, GPT-4, Llama - hammasi) **baytlardan** boshlaydi. Farqi katta.

Belgilardan boshlasak, boshlang'ich alifboni oldindan bilishimiz kerak edi -
Unicode'da 150 000 dan ortiq belgi bor. Hammasini olsak lug'at bir o'zi
to'lib ketadi; olmasak, uchramagan belgi `<unk>` bo'ladi.

Baytlardan boshlasak - alifbo **aniq 256 ta**, boshqacha bo'lishi mumkin emas.
Har qanday matn UTF-8 da baytlar ketma-ketligi:

```
"salom"   -> [115, 97, 108, 111, 109]        # 5 bayt
"salom!"  -> ...
"o'zbek"  -> ko'p baytli belgilar 2-3 baytga bo'linadi
"emoji"   -> 4 bayt
```

Shuning uchun:

- **`<unk>` fizik jihatdan mumkin emas.** Har qanday bayt - lug'atda bor.
- Alifbo hajmi doim 256 - til, emoji, kod, hech narsa buni o'zgartirmaydi.

**Bitta hiyla.** `tokenizers` kutubxonasi baytlarni ko'rinadigan belgilarga
"o'raydi", chunki 0-31 baytlarni (`\n`, tab) JSON faylda saqlab bo'lmaydi.
Shuning uchun lug'atda `Ġ` (bo'sh joy) va `Ċ` (yangi qator) kabi g'alati belgilar
ko'rasiz. Bu bug emas - `Ġthe` degani "bo'sh joy + the".

**Nega bo'sh joy tokenning bir qismi?** Chunki `" the"` va `"the"` boshqa
narsalar - biri gap o'rtasida, ikkinchisi so'z ichida. Bo'sh joyni tokenga
qo'shib yuborish decode'ni ham ideal aniq qiladi: tokenlarni ulaymiz - matn
aynan o'zi bo'lib chiqadi.

In [17]:
s = "Eagerly, a girl named Kim went hiking with her friends"
print("Matn   :", s)
print("Baytlar:", list(s.encode("utf-8")))
print("Bayt soni:", len(s.encode("utf-8")))
print()

byte_alphabet = pre_tokenizers.ByteLevel.alphabet()
print("Byte-level alifbo hajmi:", len(byte_alphabet), "  <-- doim 256")
print("Namuna:", sorted(byte_alphabet)[:20])

Matn   : Eagerly, a girl named Kim went hiking with her friends
Baytlar: [69, 97, 103, 101, 114, 108, 121, 44, 32, 97, 32, 103, 105, 114, 108, 32, 110, 97, 109, 101, 100, 32, 75, 105, 109, 32, 119, 101, 110, 116, 32, 104, 105, 107, 105, 110, 103, 32, 119, 105, 116, 104, 32, 104, 101, 114, 32, 102, 114, 105, 101, 110, 100, 115]
Bayt soni: 54

Byte-level alifbo hajmi: 256   <-- doim 256
Namuna: ['!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4']


## 6. Vocabulary size

Bu bitta raqam - lekin u modelning hajmiga ham, tezligiga ham, sifatiga ham
bevosita ta'sir qiladi.

**Ikki tomonlama savdo (trade-off):**

| `vocab_size` kichik | `vocab_size` katta |
|---|---|
| embedding jadvali arzon | embedding jadvali qimmat |
| ketma-ketlik **uzun** (ko'p token) | ketma-ketlik **qisqa** |
| attention $O(n^2)$ qimmatlashadi | softmax $O(V)$ qimmatlashadi |
| har bir token ko'p marta ko'riladi (yaxshi o'rganadi) | nodir tokenlar kam ko'riladi (yomon o'rganadi) |

**Formula.** Embedding + output head parametrlari:

$$P_{\text{vocab}} = V \times d_{\text{model}} \times \begin{cases} 1 & \text{tied weights} \\ 2 & \text{alohida head} \end{cases}$$

Kichik modelda bu **umumiy parametrlarning katta ulushi** bo'lib qolishi mumkin -
quyida sanaymiz.

**Amaliy qoida:** korpus qancha kichik bo'lsa, lug'at ham shuncha kichik bo'lsin.
Har bir token embedding'ining o'rganishi uchun uni ming marta ko'rish kerak.
2M ta soddalashtirilgan bolalar hikoyasi uchun **8000** yetarli: bu korpusning
o'z lug'ati ham unchalik boy emas.

**Qattiq chegara:** `vocab_size < 65 536`, aks holda `uint16` da saqlay
olmaymiz (B qismida ko'ramiz).

In [18]:
D_MODEL = 384      # keyingi darsda quradigan modelimizning kengligi

print("d_model = {}, tied weights (embedding = output head)\n".format(D_MODEL))
head = "{:>10}{:>18}{:>16}".format("vocab", "embed params", "ulush (30M da)")
print(head)
print("-" * len(head))
for v in [1_000, 4_000, 8_000, 16_000, 32_000, 50_257, 65_535]:
    p = v * D_MODEL
    mark = "   <-- bizniki" if v == VOCAB_SIZE else ""
    print("{:>10,}{:>18,}{:>15.1%}{}".format(v, p, p / 30_000_000, mark))

print()
print("50257 - GPT-2 ning lug'ati. 65535 - uint16 dagi chegara.")

d_model = 384, tied weights (embedding = output head)

     vocab      embed params  ulush (30M da)
--------------------------------------------
     1,000           384,000           1.3%
     4,000         1,536,000           5.1%
     8,000         3,072,000          10.2%   <-- bizniki
    16,000         6,144,000          20.5%
    32,000        12,288,000          41.0%
    50,257        19,298,688          64.3%
    65,535        25,165,440          83.9%

50257 - GPT-2 ning lug'ati. 65535 - uint16 dagi chegara.


## 7. Special tokens

Special token - matndan kelib chiqmaydigan, biz **o'zimiz** qo'shadigan token.
Ular BPE merge'lariga qatnashmaydi va hech qachon bo'laklarga bo'linmaydi.

Bizga hozir **bittasi** kerak:

**`<eos>` - end of sequence.** Hujjatlar chegarasini bildiradi. Busiz `train.bin`
uzluksiz oqim bo'lib qoladi va model bir hikoyaning oxirini keyingisining boshi
deb o'rganadi - hikoyani hech qachon tugatmaydigan model chiqadi. Generatsiyada
`<eos>` - to'xtash signali.

**Nega qolganlari yo'q:**

- **`<unk>` kerak emas** - byte-level BPE da noma'lum token fizik jihatdan
  bo'lmaydi (5-bo'lim). Uni qo'shish - hech qachon ishlatilmaydigan embedding
  qatorini bekorga saqlash.
- **`<pad>` kerak emas** - biz *sequence packing* ishlatamiz (B qismi): barcha
  hujjatlar bitta uzun oqimga ulanadi va undan bir xil uzunlikdagi bo'laklar
  kesiladi. Padding yo'q -> `<pad>` ham kerak emas.
- **`<bos>` kerak emas** - oldingi hujjatning `<eos>` i keyingisining boshlanish
  signali bo'lib xizmat qiladi.
- **Chat tokenlar** (`<|user|>`, `<|assistant|>`) - bu **pretraining**, chat
  emas. Instruction tuning bosqichida qo'shiladi.

> Har bir keraksiz special token = o'lik embedding qatori + bitta chalkashlik
> manbai. Kerak bo'lganini qo'shing, "ehtimol keyin kerak bo'lar" degani emas.

**Muhim:** special tokenlar ro'yxatning **boshida** turadi, ya'ni `<eos>` ning
ID si `0`. Bu tasodif emas - biz uni keyin doim `meta.json` dan o'qiymiz,
hech qachon kodga `0` deb yozmaymiz.

## 8. Tokenizerni o'rgatamiz

Endi haqiqiy tokenizer. To'rt qism:

| Qism | Vazifasi |
|---|---|
| `models.BPE()` | algoritm - merge rules saqlanadigan joy |
| `pre_tokenizer` | matnni merge'dan **oldin** bo'laklarga ajratadi (bo'sh joy chegaralari) |
| `decoder` | teskari yo'l: tokenlardan matnga |
| `trainer` | o'rgatish sozlamalari (vocab_size, special tokens...) |

`add_prefix_space=False` - matn boshiga sun'iy bo'sh joy qo'shilmasin, aks holda
round-trip test aniq bo'lmaydi.

`initial_alphabet=ByteLevel.alphabet()` - **muhim**. Bu 256 ta baytning
hammasini lug'atga majburan kiritadi, hatto korpusda uchramagani ham. Busiz
uchramagan bayt `<unk>` bo'lib qolar edi va biz byte-level BPE ning butun
foydasini yo'qotgan bo'lardik.

In [19]:
def tokenizer_corpus_iterator(texts, batch_size=1000):
    """train_from_iterator hujjatlarni bo'lakma-bo'lak oladi -
    butun korpusni RAM ga sig'dirish shart emas."""
    for i in range(0, len(texts), batch_size):
        yield from texts[i:i + batch_size]


# Merge statistikasi to'yingani uchun namunada o'rgatamiz
train_texts_for_tokenizer = docs[:TOKENIZER_SAMPLE_DOCS]
print("Tokenizer o'rgatiladigan hujjatlar: {:,}".format(len(train_texts_for_tokenizer)))

# --- 1. Tokenizer skeleti ---
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder       = decoders.ByteLevel()

# --- 2. Trainer ---
trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),   # 256 baytning hammasi
    min_frequency=MIN_FREQUENCY,
    show_progress=True,
)

# --- 3. O'rgatish ---
t0 = time.time()
tokenizer.train_from_iterator(
    tokenizer_corpus_iterator(train_texts_for_tokenizer),
    trainer=trainer,
    length=len(train_texts_for_tokenizer),
)
print("\nO'rgatish vaqti: {:.1f} soniya".format(time.time() - t0))

# --- 4. Saqlaymiz ---
tokenizer.save(str(TOKENIZER_FILE))
print("Saqlandi:", TOKENIZER_FILE.resolve())
print("Fayl hajmi: {:.2f} MB".format(TOKENIZER_FILE.stat().st_size / 1024**2))
print("Haqiqiy vocab_size:", tokenizer.get_vocab_size())

Tokenizer o'rgatiladigan hujjatlar: 10,000

O'rgatish vaqti: 1.7 soniya
Saqlandi: C:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\5 dars\my_LLM\tokenizer\tokenizer.json
Fayl hajmi: 0.52 MB
Haqiqiy vocab_size: 8000


### Lug'atning ichiga qaraymiz

Tokenizer nimani o'rgangan? Bu katak - darsning eng foydali qismlaridan biri:
lug'at korpusning ko'zgusi. Agar u yerda `Ġthe`, `Ġsaid`, `Ġhappy` ko'rsangiz -
pipeline to'g'ri ishlagan. Agar `</div>` yoki `Gutenberg` ko'rsangiz - kechagi
tozalash yetarli bo'lmagan.

In [20]:
vocab = tokenizer.get_vocab()                       # {token_str: id}
id_to_token = {i: t for t, i in vocab.items()}

print("Birinchi 20 ta ID (special + baytlar):")
print([id_to_token[i] for i in range(20)])
print()

# Special tokenlar qayerda?
for tok in SPECIAL_TOKENS:
    print("{:>8} -> ID {}".format(tok, tokenizer.token_to_id(tok)))
print()

# Birinchi o'rganilgan merge'lar (256 + len(special) dan keyin boshlanadi)
start = 256 + len(SPECIAL_TOKENS)
print("Birinchi 40 ta O'RGANILGAN token (Ġ = bo'sh joy):")
print([id_to_token[i] for i in range(start, start + 40)])
print()

longest = sorted(vocab.keys(), key=len, reverse=True)[:15]
print("Eng uzun 15 ta token:", longest)

Birinchi 20 ta ID (special + baytlar):
['<eos>', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3']

   <eos> -> ID 0

Birinchi 40 ta O'RGANILGAN token (Ġ = bo'sh joy):
['he', 'Ġt', 'Ġa', 'Ġs', 'ed', 'Ġthe', 'in', 'Ġw', 'Ġf', 're', 'nd', 'Ġb', 'ou', 'Ġhe', 'er', 'Ġc', 'ing', 'Ġo', 'Ġl', 'Ġh', 'it', 'ar', 'Ġto', 'at', 'Ġand', 'Ġd', 'is', 'an', 'The', 'Ġm', 'en', 'as', 'll', 'ĠThe', 'or', 'es', 'Ġp', 'Ġg', 'el', 'gh']

Eng uzun 15 ta token: ['Ġconstellations', 'Ġcontradictions', 'Ġresponsibility', 'Ġtransformation', 'ĠResponsibility', 'Ġdisappointment', 'Ġcontradiction', 'Ġunderstanding', 'Ġextraordinary', 'Ġpossibilities', 'Ġdetermination', 'Ġunforgettable', 'Ġencouragement', 'Ġtogetherness', 'Ġpresentation']


## 9. Round-trip test

Eng arzon va eng muhim tekshiruv:

$$\text{decode}(\text{encode}(x)) \stackrel{?}{=} x$$

Agar bu tenglik buzilsa - modelingiz o'rgangan har bir tokenda mayda buzilish
bor va siz buni loss egri chizig'ida **hech qachon ko'rmaysiz**. Model
o'rganadi, loss tushadi, generatsiya esa g'alati bo'ladi.

Byte-level BPE da bu tenglik **aniq** (lossless) bo'lishi shart - taxminan emas.

Tekshiramiz: oddiy matn, ko'p baytli belgilar, emoji, bo'sh joylar,
yangi qatorlar, raqamlar va - eng muhimi - **korpusdan tashqaridagi** matn.

In [22]:
def round_trip(text, verbose=True):
    enc = tokenizer.encode(text)
    back = tokenizer.decode(enc.ids)
    ok = (back == text)
    if verbose:
        status = "OK  " if ok else "XATO"
        print("[{}] {:>3} token  {!r}".format(status, len(enc.ids), text[:60]))
        if not ok:
            print("      kutilgan: {!r}".format(text))
            print("      olingan : {!r}".format(back))
    return ok


edge_cases = [

    "The clever fox jumped over the fence.",

    "Assalomu alaykum, do‘stlar! café",          # O'zbekcha belgilar va 2–3 baytli UTF-8 belgilar

    "robot: \U0001F916\U0001F30D",                # 4 baytli emojilar 🤖🌍

    "こんにちは, ñöç",                            # Yaponcha va aksentli lotin harflari

    "   bugun    havo   juda   yaxshi   ",       # Ketma-ket va chetdagi bo'sh joylar

    "birinchi satr\nikkinchi satr\n\noxirgi satr", # Yangi qatorlar

    "\tmatn tab bilan boshlandi",                 # Tab belgisi

    "987654321 va 2.71828",                       # Butun va o'nli sonlar

    "",                                           # Bo'sh satr

    "bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb",            # Bitta belgining ko'p takrorlanishi

]

print("--- Chekka holatlar ---")
edge_ok = sum(round_trip(t) for t in edge_cases)
print("\n{}/{} o'tdi".format(edge_ok, len(edge_cases)))

assert edge_ok == len(edge_cases), "Chekka holatda round-trip buzildi!"

--- Chekka holatlar ---
[OK  ]   8 token  'The clever fox jumped over the fence.'
[OK  ]  22 token  'Assalomu alaykum, do‘stlar! café'
[OK  ]  12 token  'robot: 🤖🌍'
[OK  ]  23 token  'こんにちは, ñöç'
[OK  ]  23 token  '   bugun    havo   juda   yaxshi   '
[OK  ]  24 token  'birinchi satr\nikkinchi satr\n\noxirgi satr'
[OK  ]  14 token  '\tmatn tab bilan boshlandi'
[OK  ]  19 token  '987654321 va 2.71828'
[OK  ]   0 token  ''
[OK  ]  16 token  'bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb'

10/10 o'tdi


### Bitta istisno: matn ichida yozilgan special token

Yuqoridagi test o'tdi. Endi ataylab chalkash holatni ko'ramiz: foydalanuvchi
matnining o'zida `<eos>` so'zi yozilgan bo'lsa nima bo'ladi?

In [23]:
tricky = "<eos> matn ichida yozilgan"
enc = tokenizer.encode(tricky)

print("Matn      :", repr(tricky))
print("Tokenlar  :", enc.tokens[:5], "...")
print("IDs       :", enc.ids[:5], "...")
print()
print("decode() default            :", repr(tokenizer.decode(enc.ids)))
print("decode(skip_special=False)  :",
      repr(tokenizer.decode(enc.ids, skip_special_tokens=False)))

Matn      : '<eos> matn ichida yozilgan'
Tokenlar  : ['<eos>', 'Ġmat', 'n', 'Ġ', 'ich'] ...
IDs       : [0, 1608, 78, 221, 2165] ...

decode() default            : ' matn ichida yozilgan'
decode(skip_special=False)  : '<eos> matn ichida yozilgan'


Nima bo'ldi? `<eos>` matn sifatida emas, **haqiqiy special token** (ID 0) bo'lib
kodlandi. `decode()` esa sukut bo'yicha `skip_special_tokens=True` - u special
tokenlarni chiqarib tashlaydi, chunki generatsiyada foydalanuvchiga `<eos>`
ko'rsatilmasligi kerak.

Ikkita xulosa:

1. **Bu bug emas, xususiyat.** Shuning uchun special tokenlar `<...>` kabi
   matnda deyarli uchramaydigan shaklda yoziladi (GPT-2 da yanada qattiqroq:
   `<|endoftext|>`).
2. **Foydalanuvchi matni hech qachon xom holda special tokenga aylanmasligi
   kerak** - aks holda kimdir prompt ichiga `<eos>` yozib, modelning
   generatsiyasini majburan to'xtata oladi. Bu prompt injection'ning eng
   sodda ko'rinishi. Bizning pretraining uchun muammo emas, lekin chat
   modelida bu jiddiy xavfsizlik masalasi.

In [24]:
# Korpusning O'RGATISHDA QATNASHMAGAN qismidan tekshiramiz (held-out)
held_out = docs[TOKENIZER_SAMPLE_DOCS:] or docs[-1000:]
check_sample = random.sample(held_out, min(2000, len(held_out)))

failures = [t for t in check_sample if tokenizer.decode(tokenizer.encode(t).ids) != t]

print("Tekshirilgan hujjatlar : {:,}".format(len(check_sample)))
print("Xatolar                : {}".format(len(failures)))

assert not failures, "Round-trip buzildi! Tokenizer sozlamalarini tekshiring."
print("\nROUND-TRIP TEST O'TDI - encode/decode lossless.")

Tekshirilgan hujjatlar : 1,000
Xatolar                : 0

ROUND-TRIP TEST O'TDI - encode/decode lossless.


## 10. Tokenizatsiyani o'lchash: fertility va compression

Tokenizer "yaxshi" ekanini his-tuyg'u bilan emas, ikkita raqam bilan aytamiz.

**Fertility** - bitta so'zga nechta token to'g'ri keladi:

$$\text{fertility} = \frac{\text{tokenlar soni}}{\text{so'zlar soni}}, \qquad \textbf{past} = \textbf{yaxshi}$$

**Compression** - bitta tokenga nechta belgi to'g'ri keladi:

$$\text{compression} = \frac{\text{belgilar soni}}{\text{tokenlar soni}}, \qquad \textbf{yuqori} = \textbf{yaxshi}$$

Ikkalasi bir xil narsani ikki tomondan o'lchaydi. Ingliz tili uchun odatiy
qiymatlar: fertility 1.2-1.5, compression ~4 belgi/token.

**Ikkita qoida:**

1. **Held-out sample'da o'lchanadi** - o'rgatilgan matnda emas. O'rgatish
   matnida har qanday tokenizer yaxshi ko'rinadi, chunki merge'lar aynan o'sha
   matnga moslashtirilgan.
2. **Bir xil matnda solishtiriladi** - GPT-2 ni bir matnda, o'zimiznikini
   boshqasida o'lchash hech narsa demaydi.

**Nega bu muhim.** Compression 4 dan 3 ga tushsa, bir xil matn uchun 33% ko'proq
token kerak bo'ladi: training 33% qimmat, context window 25% qisqa. Bu bitta
raqam - butun loyihaning byudjeti.

In [25]:
def measure(tok, texts):
    n_tokens = sum(len(e.ids) for e in tok.encode_batch(texts))
    n_words  = sum(len(t.split()) for t in texts)
    n_chars  = sum(len(t) for t in texts)
    n_bytes  = sum(len(t.encode("utf-8")) for t in texts)
    return {
        "tokens": n_tokens,
        "fertility": n_tokens / n_words,
        "compression": n_chars / n_tokens,
        "bytes_per_token": n_bytes / n_tokens,
    }


eval_sample = random.sample(held_out, min(5000, len(held_out)))   # HELD-OUT
ours = measure(tokenizer, eval_sample)

print("Baholash namunasi: {:,} hujjat (o'rgatishda qatnashmagan)\n".format(len(eval_sample)))
print("Bizning tokenizer (vocab={:,})".format(tokenizer.get_vocab_size()))
print("  fertility       : {:.3f}  token/so'z     (past = yaxshi)".format(ours["fertility"]))
print("  compression     : {:.3f}  belgi/token    (yuqori = yaxshi)".format(ours["compression"]))
print("  bytes per token : {:.3f}".format(ours["bytes_per_token"]))

Baholash namunasi: 1,000 hujjat (o'rgatishda qatnashmagan)

Bizning tokenizer (vocab=8,000)
  fertility       : 1.260  token/so'z     (past = yaxshi)
  compression     : 4.370  belgi/token    (yuqori = yaxshi)
  bytes per token : 4.370


In [26]:
# GPT-2 bilan solishtirish - AYNAN SHU matnda.
# Internet kerak; bo'lmasa bu katak sakraladi.
try:
    gpt2 = Tokenizer.from_pretrained("gpt2")
    theirs = measure(gpt2, eval_sample)

    head = "{:<22}{:>12}{:>14}{:>14}".format("Tokenizer", "vocab", "fertility", "compression")
    print(head)
    print("-" * len(head))
    print("{:<22}{:>12,}{:>14.3f}{:>14.3f}".format(
        "bizniki", tokenizer.get_vocab_size(), ours["fertility"], ours["compression"]))
    print("{:<22}{:>12,}{:>14.3f}{:>14.3f}".format(
        "GPT-2", gpt2.get_vocab_size(), theirs["fertility"], theirs["compression"]))
    print()
    rel = (ours["tokens"] - theirs["tokens"]) / theirs["tokens"]
    ratio = gpt2.get_vocab_size() / tokenizer.get_vocab_size()
    print("Bir xil matn uchun bizniki GPT-2 ga nisbatan {:+.1%} token ishlatadi".format(rel))
    print("Lug'atimiz esa {:.0f} barobar kichik.".format(ratio))
    print()
    if rel <= 0:
        print("Bizniki YAXSHIROQ siqmoqda - lug'ati kichik bo'lsa ham. Sababi:")
        print("GPT-2 butun internetga o'rgatilgan, biz esa aynan shu korpusga.")
        print("Tor domen -> har bir slot foydali -> samarali lug'at.")
    else:
        print("Bizniki biroz ko'proq token ishlatmoqda, lekin lug'ati ancha kichik.")
        print("Bu kutilgan savdo: vocab_size ni oshirsangiz farq kamayadi.")
        print("VOCAB_SIZE ni {} dan oshirib ko'ring - vazifaning 1-topshirig'i.".format(
            tokenizer.get_vocab_size()))
except Exception as e:
    print("GPT-2 tokenizerini yuklab bo'lmadi (internet yo'qmi?):", e)

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

C:\Users\xolmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\xolmu\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Tokenizer                    vocab     fertility   compression
--------------------------------------------------------------
bizniki                      8,000         1.260         4.370
GPT-2                       50,257         1.271         4.330

Bir xil matn uchun bizniki GPT-2 ga nisbatan -0.9% token ishlatadi
Lug'atimiz esa 6 barobar kichik.

Bizniki YAXSHIROQ siqmoqda - lug'ati kichik bo'lsa ham. Sababi:
GPT-2 butun internetga o'rgatilgan, biz esa aynan shu korpusga.
Tor domen -> har bir slot foydali -> samarali lug'at.


In [28]:
# Bir xil jumla ikkala tokenizerda qanday bo'linadi
demo = "Eagerly, a girl named Kim went hiking with her friends. They found a secret passageway under a big rock."

enc = tokenizer.encode(demo)
print("BIZNIKI  ({} token):".format(len(enc.ids)))
print(" ", enc.tokens)
print("  IDs:", enc.ids)
print()

try:
    g = gpt2.encode(demo)
    print("GPT-2    ({} token):".format(len(g.ids)))
    print(" ", g.tokens)
except NameError:
    pass

BIZNIKI  (24 token):
  ['Eagerly', ',', 'Ġa', 'Ġgirl', 'Ġnamed', 'ĠKim', 'Ġwent', 'Ġhiking', 'Ġwith', 'Ġher', 'Ġfriends', '.', 'ĠThey', 'Ġfound', 'Ġa', 'Ġsecret', 'Ġpass', 'age', 'way', 'Ġunder', 'Ġa', 'Ġbig', 'Ġrock', '.']
  IDs: [2800, 12, 259, 476, 587, 611, 1346, 7615, 333, 307, 482, 14, 352, 440, 259, 837, 2144, 641, 523, 642, 259, 628, 941, 14]

GPT-2    (26 token):
  ['E', 'ager', 'ly', ',', 'Ġa', 'Ġgirl', 'Ġnamed', 'ĠKim', 'Ġwent', 'Ġhiking', 'Ġwith', 'Ġher', 'Ġfriends', '.', 'ĠThey', 'Ġfound', 'Ġa', 'Ġsecret', 'Ġpass', 'ag', 'eway', 'Ġunder', 'Ġa', 'Ġbig', 'Ġrock', '.']


In [29]:
# Tokenizer nimani YOMON bo'ladi? Domendan tashqaridagi matn.
out_of_domain = [
    "Once upon a time a little rabbit found a red ball.",   # o'z domenimiz
    "def encode(self, text: str) -> List[int]: return self._bpe(text)",
    "Kecha men O'zbekistonda bo'ldim va juda ko'p narsa o'rgandim.",
    "The mitochondrion is the powerhouse of the eukaryotic cell.",
]

head = "{:>12}{:>12}  {}".format("tokenlar", "belgi/token", "matn")
print(head)
print("-" * 70)
for t in out_of_domain:
    n = len(tokenizer.encode(t).ids)
    print("{:>12}{:>12.2f}  {}".format(n, len(t) / n, t[:44]))

print()
print("Bolalar hikoyasi eng samarali siqiladi - lug'at aynan shunga o'rgatilgan.")
print("Kod va o'zbek tili yomon siqiladi: model ularni ko'rmagan.")
print("Xulosa: tokenizer - korpusning ko'zgusi, universal narsa emas.")

    tokenlar belgi/token  matn
----------------------------------------------------------------------
          12        4.17  Once upon a time a little rabbit found a red
          32        2.00  def encode(self, text: str) -> List[int]: re
          35        1.74  Kecha men O'zbekistonda bo'ldim va juda ko'p
          21        2.81  The mitochondrion is the powerhouse of the e

Bolalar hikoyasi eng samarali siqiladi - lug'at aynan shunga o'rgatilgan.
Kod va o'zbek tili yomon siqiladi: model ularni ko'rmagan.
Xulosa: tokenizer - korpusning ko'zgusi, universal narsa emas.


## 11. `train_tokenizer.py`

Notebook - o'rganish uchun. Loyihada esa tokenizer **skript** bilan qayta
tiklanadigan bo'lishi kerak: `tokenizer.json` `.gitignore` da emas, lekin uni
qayta yaratadigan kod ham kerak. Loyiha qoidasi (`README.md`):

> Artefaktni emas, uni qayta yaratadigan kodni commit qilamiz.

Quyidagi katak `our-llm/tokenizer/train_tokenizer.py` faylini yozadi -
yuqoridagi mantiqning aynan o'zi, CLI ko'rinishida.

In [30]:
%%writefile train_tokenizer.py
"""Byte-level BPE tokenizerni final korpusdan o'rgatadi.

Ishlatish:
    python tokenizer/train_tokenizer.py \
        --input  data/final/tinystories_final.jsonl \
        --output tokenizer/tokenizer.json \
        --vocab-size 8000
"""
import argparse
import json
from pathlib import Path

from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers

SPECIAL_TOKENS = ["<eos>"]


def iter_texts(path, limit=None):
    """JSONL fayldan matnlarni oqim sifatida o'qiydi (RAM ga sig'dirmasdan)."""
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit:
                return
            yield json.loads(line)["text"]


def build_tokenizer(vocab_size, min_frequency):
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()

    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=SPECIAL_TOKENS,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        min_frequency=min_frequency,
        show_progress=True,
    )
    return tok, trainer


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--input", required=True, type=Path)
    p.add_argument("--output", required=True, type=Path)
    p.add_argument("--vocab-size", type=int, default=8000)
    p.add_argument("--min-frequency", type=int, default=2)
    p.add_argument("--max-docs", type=int, default=200_000,
                   help="BPE statistikasi tez to'yinadi; None uchun 0 bering")
    args = p.parse_args()

    if args.vocab_size >= 65_536:
        raise SystemExit("vocab_size < 65536 bo'lishi kerak (uint16 dataset uchun)")

    limit = args.max_docs or None
    tok, trainer = build_tokenizer(args.vocab_size, args.min_frequency)
    tok.train_from_iterator(iter_texts(args.input, limit), trainer=trainer)

    # Round-trip - saqlashdan OLDIN. Buzilgan tokenizer diskka chiqmasin.
    for probe in ["Once upon a time.", "  bo'sh   joylar ", "qator\nqator", "salom!"]:
        back = tok.decode(tok.encode(probe).ids)
        if back != probe:
            raise SystemExit("Round-trip xato: %r -> %r" % (probe, back))

    args.output.parent.mkdir(parents=True, exist_ok=True)
    tok.save(str(args.output))
    print("Saqlandi: %s  (vocab_size=%d)" % (args.output, tok.get_vocab_size()))


if __name__ == "__main__":
    main()

Writing train_tokenizer.py


In [31]:
# Skript ishlayaptimi - tekshiramiz (kichik namunada, tez)
!python train_tokenizer.py --input ../data/final/tinystories_final.jsonl --output ../dataset/_smoke_tokenizer.json --vocab-size 1000 --max-docs 5000

Traceback (most recent call last):
  File "c:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\5 dars\my_LLM\train_tokenizer.py", line 71, in <module>
    main()
  File "c:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\5 dars\my_LLM\train_tokenizer.py", line 57, in main
    tok.train_from_iterator(iter_texts(args.input, limit), trainer=trainer)
  File "c:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\5 dars\my_LLM\train_tokenizer.py", line 20, in iter_texts
    with open(path, "r", encoding="utf-8") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\final\\tinystories_final.jsonl'


---

# B. DATASET BUILDER

Tokenizer tayyor. Endi butun korpusni bir marta kodlab, training loop tez o'qiy
oladigan formatda diskka yozamiz.

**Nega bu alohida bosqich?** Training paytida matnni har epoxada qayta
tokenizatsiya qilish - bekorga sarflangan CPU. Bir marta kodlaymiz, natijani
`train.bin` / `val.bin` ga yozamiz, keyin training loop shunchaki xotiradan
raqam o'qiydi.

Nima quramiz:

```
docs (matn)
   |  1. SPLIT  (kodlashdan OLDIN!)
   +--> train docs                  val docs
   |       |  2. encode + <eos>        |
   |       v                           v
   |    [7355, 1029, ..., 0]        [412, 88, ..., 0]
   |       |  3. uint16 binary         |
   |       v                           v
   |    train.bin                   val.bin
   |
   +--> 4. meta.json  (vocab_size, block_size, eos_id, token soni)
```

## 12. Split BEFORE encoding

Eng ko'p uchraydigan xato: hamma narsani kodlab, keyin token oqimini kesish.

**Nega bu xato:**

1. **Hujjat ikkiga bo'linib ketadi.** Bitta hikoyaning boshi train'da, oxiri
   val'da qoladi. Model train'da boshini ko'radi, val'da davomini bashorat
   qiladi - val loss soxta darajada past chiqadi. Bu **data leakage**.
2. **Val split takrorlanmaydi.** Tokenizer o'zgarsa, kesish nuqtasi ham
   siljiydi va kechagi val loss bugungisi bilan solishtirib bo'lmaydi.

**To'g'ri tartib:** avval **hujjat darajasida** bo'lamiz, keyin har bir qismni
alohida kodlaymiz. Shunda bitta hikoya butunligicha bitta tomonda qoladi.

Aralashtirish `SEED` bilan - takrorlanadigan bo'lishi uchun.

In [32]:
indices = list(range(len(docs)))
rng = random.Random(SEED)          # aynan takrorlanadigan aralashtirish
rng.shuffle(indices)

n_val = max(1, int(len(docs) * VAL_RATIO))
val_idx   = indices[:n_val]
train_idx = indices[n_val:]

train_docs = [docs[i] for i in train_idx]
val_docs   = [docs[i] for i in val_idx]

print("Train hujjatlar : {:,}".format(len(train_docs)))
print("Val hujjatlar   : {:,}  ({:.2%})".format(len(val_docs), len(val_docs) / len(docs)))
print()

# Tekshirish: bir xil hujjat ikkala tomonda bo'lmasin
assert set(train_idx).isdisjoint(set(val_idx)), "Split kesishib ketdi!"
assert len(train_idx) + len(val_idx) == len(docs), "Hujjatlar yo'qoldi!"
print("Split toza: kesishuv yo'q, hujjat yo'qolmadi.")

Train hujjatlar : 9,900
Val hujjatlar   : 100  (1.00%)

Split toza: kesishuv yo'q, hujjat yo'qolmadi.


## 13. Nega `uint16`

Har bir token - bitta butun son. Uni diskda qanday saqlaymiz?

| dtype | bir token | 500M token | Chegara |
|---|---|---|---|
| `int64` | 8 bayt | **4.0 GB** | juda katta |
| `int32` | 4 bayt | 2.0 GB | hali ham isrof |
| **`uint16`** | **2 bayt** | **1.0 GB** | `0 .. 65535` |
| `uint8` | 1 bayt | 0.5 GB | `0..255` - yetmaydi |

Bizning eng katta token ID si $8000 - 1 = 7999$, ya'ni `uint16` ga bemalol
sig'adi. `int64` ishlatish - bir xil ma'lumot uchun **4 barobar** ko'p disk va
4 barobar ko'p disk I/O.

**Qattiq shart:**

$$\text{vocab\_size} \le 65\,536$$

Aynan shuning uchun 6-bo'limda lug'at hajmini tanlashda bu chegarani aytdik.

**Nega umuman "binary"?** `np.memmap` diskdagi faylni xotiraga yuklanmagan
holda massiv kabi ko'rsatadi. Operatsion tizim faqat kerakli sahifani o'qiydi.
Natijada 10 GB lik dataset'dan 8 GB RAM li mashinada ham batch olish mumkin.
Buning uchun fayl **bir xil o'lchamli sonlar ketma-ketligi** bo'lishi shart -
JSON yoki CSV emas.

In [33]:
DTYPE = np.uint16

assert tokenizer.get_vocab_size() <= np.iinfo(DTYPE).max + 1, \
    "vocab_size uint16 ga sig'maydi!"

print("dtype              :", np.dtype(DTYPE).name)
print("Bir token          :", np.dtype(DTYPE).itemsize, "bayt")
print("Maksimal ID        :", np.iinfo(DTYPE).max)
print("Bizning max ID     :", tokenizer.get_vocab_size() - 1)
print("Zaxira             : {:,} ta bo'sh joy".format(
    np.iinfo(DTYPE).max - tokenizer.get_vocab_size() + 1))

dtype              : uint16
Bir token          : 2 bayt
Maksimal ID        : 65535
Bizning max ID     : 7999
Zaxira             : 57,536 ta bo'sh joy


## 14. Kodlash va `.bin` ga yozish

Har bir hujjat oxiriga `<eos>` qo'shamiz:

```
[hikoya 1 tokenlari] <eos> [hikoya 2 tokenlari] <eos> [hikoya 3 tokenlari] <eos> ...
```

Diqqat qiladigan uchta narsa:

- **`encode_batch`** - bitta-bitta emas, partiyalab kodlaymiz. Kutubxona ichida
  Rust va ko'p yadro ishlaydi, bu ~10 barobar tezroq.
- **Oqim bilan yozamiz** - butun natijani RAM da yig'maymiz. `arr.tofile(f)` har
  partiyani darhol diskka tushiradi, shuning uchun korpus RAM dan katta
  bo'lsa ham ishlaydi.
- **`eos_id` ni `token_to_id` dan olamiz** - kodga `0` deb yozmaymiz. Special
  tokenlar ro'yxati o'zgarsa, kod o'zi moslashadi.

In [34]:
eos_id = tokenizer.token_to_id("<eos>")
assert eos_id is not None, "<eos> tokeni topilmadi!"
print("eos_id =", eos_id)


def encode_to_bin(texts, out_path, batch_size=2000):
    """Hujjatlarni kodlab, uint16 binary fayl sifatida yozadi.
    Har bir hujjatdan keyin <eos> qo'shiladi."""
    n_tokens = 0
    with open(out_path, "wb") as f:
        for i in tqdm(range(0, len(texts), batch_size),
                      desc=out_path.name, unit="batch"):
            batch = texts[i:i + batch_size]
            encodings = tokenizer.encode_batch(batch)

            ids = []
            for enc in encodings:
                ids.extend(enc.ids)
                ids.append(eos_id)          # hujjat chegarasi

            arr = np.array(ids, dtype=DTYPE)
            arr.tofile(f)                   # darhol diskka
            n_tokens += arr.size
    return n_tokens


t0 = time.time()
n_train_tokens = encode_to_bin(train_docs, TRAIN_BIN)
n_val_tokens   = encode_to_bin(val_docs,   VAL_BIN)
elapsed = time.time() - t0

print()
print("Kodlash vaqti : {:.1f} soniya".format(elapsed))
print("train.bin     : {:>14,} token   {:>8.1f} MB".format(
    n_train_tokens, TRAIN_BIN.stat().st_size / 1024**2))
print("val.bin       : {:>14,} token   {:>8.1f} MB".format(
    n_val_tokens, VAL_BIN.stat().st_size / 1024**2))
print("Jami          : {:>14,} token".format(n_train_tokens + n_val_tokens))

eos_id = 0


train.bin:   0%|          | 0/5 [00:00<?, ?batch/s]

val.bin:   0%|          | 0/1 [00:00<?, ?batch/s]


Kodlash vaqti : 2.6 soniya
train.bin     :      2,804,299 token        5.3 MB
val.bin       :         28,537 token        0.1 MB
Jami          :      2,832,836 token


## 15. `meta.json`

`train.bin` - bu shunchaki raqamlar. O'zi haqida hech narsa aytmaydi: qaysi
tokenizer yaratgan? `<eos>` ID si nechchi? dtype qanaqa?

Bu ma'lumotlarsiz fayl **o'qib bo'lmaydigan** bo'lib qoladi - va bir necha
haftadan keyin siz albatta unutasiz.

`meta.json` - datasetning pasporti. Training kodi barcha raqamlarni shu yerdan
o'qiydi, hech qachon o'zi ichiga yozmaydi.

In [35]:
meta = {
    "source": str(FINAL_FILE).replace("\\", "/"),
    "tokenizer_file": str(TOKENIZER_FILE).replace("\\", "/"),
    "vocab_size": tokenizer.get_vocab_size(),
    "special_tokens": {t: tokenizer.token_to_id(t) for t in SPECIAL_TOKENS},
    "eos_id": eos_id,
    "dtype": np.dtype(DTYPE).name,
    "block_size": BLOCK_SIZE,
    "seed": SEED,
    "val_ratio": VAL_RATIO,
    "n_docs_total": len(docs),
    "n_docs_train": len(train_docs),
    "n_docs_val": len(val_docs),
    "n_tokens_train": int(n_train_tokens),
    "n_tokens_val": int(n_val_tokens),
    "created": time.strftime("%Y-%m-%d %H:%M:%S"),
}

with open(META_FILE, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(json.dumps(meta, indent=2, ensure_ascii=False))
print()
print("Saqlandi:", META_FILE.resolve())

{
  "source": "c:/Users/xolmu/OneDrive/Desktop/Darslar oyma oy/LLM 2/5 dars/my_LLM/data/final/simplestories_final.jsonl",
  "tokenizer_file": "c:/Users/xolmu/OneDrive/Desktop/Darslar oyma oy/LLM 2/5 dars/my_LLM/tokenizer/tokenizer.json",
  "vocab_size": 8000,
  "special_tokens": {
    "<eos>": 0
  },
  "eos_id": 0,
  "dtype": "uint16",
  "block_size": 256,
  "seed": 1337,
  "val_ratio": 0.01,
  "n_docs_total": 10000,
  "n_docs_train": 9900,
  "n_docs_val": 100,
  "n_tokens_train": 2804299,
  "n_tokens_val": 28537,
  "created": "2026-10-20 22:20:16"
}

Saqlandi: C:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\5 dars\my_LLM\dataset\meta.json


## 16. Sequence packing

Model bir xil uzunlikdagi bo'laklarni kutadi (`block_size`). Hikoyalar esa
turli uzunlikda. Ikki yo'l bor:

**Padding (biz ishlatmaymiz):**

```
[hikoya 1][PAD][PAD][PAD][PAD]
[hikoya 2 - uzunroq...][PAD]
[qisqa][PAD][PAD][PAD][PAD][PAD]
```
GPU `<pad>` tokenlar ustida ham hisoblaydi. Qisqa hikoyalar ko'p bo'lsa,
hisobning yarmi bekorga ketishi mumkin.

**Packing (biz shuni ishlatamiz):**

```
... <eos> Once upon a time there was <eos> A little boy found a <eos> The dog ran ...
    |<------- block 0 ------->|<------- block 1 ------->|<---- block 2 ...
```

Hamma narsa bitta uzluksiz oqim. Bo'laklarni shunchaki kesib olamiz.
**GPU ning 100% i haqiqiy tokenga ishlaydi, padding umuman yo'q.**

Ha, ba'zi bo'laklar hikoya o'rtasidan boshlanadi va ikkita hikoya chegarasidan
o'tadi. Bu muammo emas - `<eos>` modelga chegarani ko'rsatib turadi va model
"`<eos>` dan keyin yangi hikoya boshlanadi" degan qoidani aynan shu tarzda
o'rganadi.

`.bin` faylimiz allaqachon aynan shunday uzluksiz oqim - qo'shimcha ish
qilish shart emas.

## 17. Input / target shift

Bu darsdagi eng muhim tushunchalardan biri.

Model **keyingi tokenni bashorat qilishni** o'rganadi. Demak har bir pozitsiya
uchun to'g'ri javob - **keyingi** token. Shuning uchun target = input, bir
qadam surilgan:

```
oqim:    [ The ][ cat ][ sat ][ on ][ the ][ mat ][ <eos> ]
           i      i+1    i+2    i+3   i+4    i+5     i+6

x  =     [ The ][ cat ][ sat ][ on ][ the ][ mat ]      <- data[i   : i+block]
y  =            [ cat ][ sat ][ on ][ the ][ mat ][<eos>] <- data[i+1 : i+1+block]
```

Modelning o'rganadigan juftliklari:

| model ko'radi | bashorat qilishi kerak |
|---|---|
| `The` | `cat` |
| `The cat` | `sat` |
| `The cat sat` | `on` |
| `The cat sat on` | `the` |
| ... | ... |

Bitta `block_size=256` lik bo'lakdan **256 ta o'quv misoli** chiqadi - hammasi
bir vaqtda, parallel. Buni causal mask ta'minlaydi: har bir pozitsiya faqat
o'zidan chapdagilarni ko'radi. (Buni keyingi darsda quramiz.)

**Doimiy tenglik:**

$$y[t] = x[t+1]$$

Agar bu buzilsa - model kelajakni ko'radi (loss deyarli nolga tushadi, model
esa hech narsa o'rganmagan bo'ladi) yoki noto'g'ri narsani o'rganadi.
Quyida buni tekshiramiz.

**Nega `len(data) - block_size - 1`?** Chunki `y` uchun bitta qo'shimcha
token kerak - `+1` siljish. Busiz oxirgi bo'lakda `IndexError` olasiz.

In [36]:
# memmap: fayl diskda qoladi, RAM ga to'liq yuklanmaydi
train_data = np.memmap(TRAIN_BIN, dtype=DTYPE, mode="r")
val_data   = np.memmap(VAL_BIN,   dtype=DTYPE, mode="r")

print("train_data:", train_data.shape, train_data.dtype)
print("val_data  :", val_data.shape, val_data.dtype)


def get_batch(split, batch_size=4, block_size=BLOCK_SIZE):
    """Tasodifiy joydan bo'lak kesib oladi. Training loop shu funksiyani chaqiradi."""
    data = train_data if split == "train" else val_data

    # -1: y uchun bitta qo'shimcha token kerak
    ix = np.random.randint(0, len(data) - block_size - 1, size=batch_size)

    x = np.stack([data[i     : i + block_size    ].astype(np.int64) for i in ix])
    y = np.stack([data[i + 1 : i + block_size + 1].astype(np.int64) for i in ix])
    return x, y


x, y = get_batch("train", batch_size=4)
print()
print("x shape:", x.shape, "  dtype:", x.dtype)
print("y shape:", y.shape, "  dtype:", y.dtype)
print()
print("PyTorch da bu shunchaki: torch.from_numpy(x) -> [batch, block_size]")

train_data: (2804299,) uint16
val_data  : (28537,) uint16

x shape: (4, 256)   dtype: int64
y shape: (4, 256)   dtype: int64

PyTorch da bu shunchaki: torch.from_numpy(x) -> [batch, block_size]


In [37]:
# Shift ko'z bilan ko'rinsin
xs, ys = get_batch("train", batch_size=1, block_size=12)

print("x (input) :", xs[0].tolist())
print("y (target):", ys[0].tolist())
print()
print("Tokenlar bilan:")
print("  x:", [tokenizer.id_to_token(int(i)) for i in xs[0]])
print("  y:", [tokenizer.id_to_token(int(i)) for i in ys[0]])
print()
print("Model o'rganadigan juftliklar:")
for k in range(6):
    ctx = tokenizer.decode([int(i) for i in xs[0][:k + 1]])
    nxt = tokenizer.decode([int(ys[0][k])])
    print("  {!r:<45} -> {!r}".format(ctx, nxt))

x (input) : [4213, 12, 1887, 259, 3708, 347, 1066, 331, 435, 1584, 14, 0]
y (target): [12, 1887, 259, 3708, 347, 1066, 331, 435, 1584, 14, 0, 2612]

Tokenlar bilan:
  x: ['Ġbouncing', ',', 'Ġcreating', 'Ġa', 'Ġrhythm', 'Ġthat', 'Ġonly', 'Ġthey', 'Ġcould', 'Ġhear', '.', '<eos>']
  y: [',', 'Ġcreating', 'Ġa', 'Ġrhythm', 'Ġthat', 'Ġonly', 'Ġthey', 'Ġcould', 'Ġhear', '.', '<eos>', 'Bright']

Model o'rganadigan juftliklar:
  ' bouncing'                                   -> ','
  ' bouncing,'                                  -> ' creating'
  ' bouncing, creating'                         -> ' a'
  ' bouncing, creating a'                       -> ' rhythm'
  ' bouncing, creating a rhythm'                -> ' that'
  ' bouncing, creating a rhythm that'           -> ' only'


## 18. To'rtta tekshiruv

Dataset training'ga topshirilishidan oldin to'rtta savolga javob berishi kerak.
Bu tekshiruvlar bir necha soniya oladi; ularsiz ketgan xato esa bir necha kunlik
GPU vaqtini yeydi.

In [38]:
print("=" * 62)
print("1-TEKSHIRUV: fayl hajmi va dtype")
print("=" * 62)
itemsize = np.dtype(DTYPE).itemsize
for path, n in [(TRAIN_BIN, n_train_tokens), (VAL_BIN, n_val_tokens)]:
    expected = n * itemsize
    actual = path.stat().st_size
    assert expected == actual, "{}: {} != {}".format(path.name, expected, actual)
    print("  {:<12} {:>14,} token x {} bayt = {:,} bayt  OK".format(
        path.name, n, itemsize, actual))

1-TEKSHIRUV: fayl hajmi va dtype
  train.bin         2,804,299 token x 2 bayt = 5,608,598 bayt  OK
  val.bin              28,537 token x 2 bayt = 57,074 bayt  OK


In [39]:
print("=" * 62)
print("2-TEKSHIRUV: ID diapazoni (0 <= id < vocab_size)")
print("=" * 62)
V = tokenizer.get_vocab_size()
for name, data in [("train", train_data), ("val", val_data)]:
    lo, hi = int(data.min()), int(data.max())
    assert 0 <= lo and hi < V, "{}: ID diapazondan chiqdi!".format(name)
    print("  {:<8} min={:<6} max={:<6} (vocab_size={})  OK".format(name, lo, hi, V))

# Bonus: <eos> lar soni hujjatlar soniga tengmi?
n_eos_train = int((np.asarray(train_data) == eos_id).sum())
n_eos_val   = int((np.asarray(val_data)   == eos_id).sum())
print()
print("  <eos> train: {:,}   hujjatlar: {:,}".format(n_eos_train, len(train_docs)))
print("  <eos> val  : {:,}   hujjatlar: {:,}".format(n_eos_val, len(val_docs)))
print("  (matn ichida '<eos>' yozilgan bo'lsa, biroz ko'proq chiqishi mumkin)")

2-TEKSHIRUV: ID diapazoni (0 <= id < vocab_size)
  train    min=0      max=7999   (vocab_size=8000)  OK
  val      min=0      max=7995   (vocab_size=8000)  OK

  <eos> train: 9,900   hujjatlar: 9,900
  <eos> val  : 100   hujjatlar: 100
  (matn ichida '<eos>' yozilgan bo'lsa, biroz ko'proq chiqishi mumkin)


In [40]:
print("=" * 62)
print("3-TEKSHIRUV: round-trip - .bin dan o'qib, matnga qaytarish")
print("=" * 62)
start = np.random.randint(0, len(train_data) - 200)
window = [int(i) for i in train_data[start:start + 200]]
text = tokenizer.decode(window)

print("  train.bin ning {:,}-pozitsiyasidan 200 token:\n".format(start))
print(text[:600])
print()
assert len(text) > 100, "Decode bo'sh yoki juda qisqa - nimadir buzilgan"
print("  Matn o'qilarli  OK")

3-TEKSHIRUV: round-trip - .bin dan o'qib, matnga qaytarish
  train.bin ning 1,357,896-pozitsiyasidan 200 token:

 a ghost among the living. The ocean that once held beauty now only brought pain. 

Just then, he spotted a small boy watching the waves. The child's eyes sparkled with wonder, and he asked, "Do you think the tide will bring my dad back?" The fisherman felt his heart ache but also fill with purpose. He knelt beside the boy, sharing stories of the sea. Together, they watched the waves, finding hope in the rhythm of the tide. In that moment, the fisherman understood that love could still be celebrated, even in loss.By a river, a young man named Jose discovered a rusty old watch. Curious, he woun

  Matn o'qilarli  OK


In [41]:
print("=" * 62)
print("4-TEKSHIRUV: input/target shift  ->  y[t] == x[t+1]")
print("=" * 62)
xb, yb = get_batch("train", batch_size=32, block_size=BLOCK_SIZE)

assert np.array_equal(xb[:, 1:], yb[:, :-1]), "SHIFT BUZILGAN!"
print("  x[:, 1:] == y[:, :-1]   ->  32 x {} pozitsiyada tekshirildi  OK".format(BLOCK_SIZE))
print()
print("  Bu tenglik y ning x dan aynan bir qadam oldinda ekanini isbotlaydi.")
print()
print("=" * 62)
print("TO'RTTA TEKSHIRUV HAM O'TDI - dataset training'ga tayyor.")
print("=" * 62)

4-TEKSHIRUV: input/target shift  ->  y[t] == x[t+1]
  x[:, 1:] == y[:, :-1]   ->  32 x 256 pozitsiyada tekshirildi  OK

  Bu tenglik y ning x dan aynan bir qadam oldinda ekanini isbotlaydi.

TO'RTTA TEKSHIRUV HAM O'TDI - dataset training'ga tayyor.


## 19. Yakuniy hisobot

In [42]:
report = """# Tokenizer va Dataset hisoboti

## Tokenizer

- Algoritm      : byte-level BPE
- Vocab size    : {vocab:,}
- Special tokens: {special}
- O'rgatildi    : {tdocs:,} hujjat
- Fayl          : {tfile}

## Sifat (held-out namunada)

- Fertility     : {fert:.3f} token / so'z
- Compression   : {comp:.3f} belgi / token
- Round-trip    : lossless (tekshirildi)

## Dataset

- Hujjatlar     : {ndocs:,}  ->  train {ntr:,} / val {nva:,}
- Tokenlar      : train {ttr:,} / val {tva:,}
- dtype         : {dtype} ({ibytes} bayt/token)
- Block size    : {bs}
- Disk          : {mb:.1f} MB

## Keyingi qadam

train.bin + val.bin + meta.json  ->  embedding -> attention -> transformer
""".format(
    vocab=tokenizer.get_vocab_size(),
    special=SPECIAL_TOKENS,
    tdocs=len(train_texts_for_tokenizer),
    tfile=TOKENIZER_FILE,
    fert=ours["fertility"],
    comp=ours["compression"],
    ndocs=len(docs), ntr=len(train_docs), nva=len(val_docs),
    ttr=n_train_tokens, tva=n_val_tokens,
    dtype=np.dtype(DTYPE).name,
    ibytes=np.dtype(DTYPE).itemsize,
    bs=BLOCK_SIZE,
    mb=(TRAIN_BIN.stat().st_size + VAL_BIN.stat().st_size) / 1024**2,
)

report_path = DATASET_DIR / "tokenizer_report.md"
report_path.write_text(report, encoding="utf-8")
print(report)
print("Saqlandi:", report_path.resolve())

# Tokenizer va Dataset hisoboti

## Tokenizer

- Algoritm      : byte-level BPE
- Vocab size    : 8,000
- Special tokens: ['<eos>']
- O'rgatildi    : 10,000 hujjat
- Fayl          : c:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\5 dars\my_LLM\tokenizer\tokenizer.json

## Sifat (held-out namunada)

- Fertility     : 1.260 token / so'z
- Compression   : 4.370 belgi / token
- Round-trip    : lossless (tekshirildi)

## Dataset

- Hujjatlar     : 10,000  ->  train 9,900 / val 100
- Tokenlar      : train 2,804,299 / val 28,537
- dtype         : uint16 (2 bayt/token)
- Block size    : 256
- Disk          : 5.4 MB

## Keyingi qadam

train.bin + val.bin + meta.json  ->  embedding -> attention -> transformer

Saqlandi: C:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\5 dars\my_LLM\dataset\tokenizer_report.md


In [44]:
# Smoke-test faylini tozalaymiz
smoke = DATASET_DIR / "_smoke_tokenizer.json"
if smoke.exists():
    smoke.unlink()
    print("O'chirildi:", smoke.name)

print()
print("my_LLM/")
print("  data/final/tinystories_final.jsonl   <- kirish")
print("  tokenizer/")
print("    tokenizer.json                     <- lug'at + merge rules")
print("    train_tokenizer.py                 <- qayta yaratadigan skript")
print("    tokenizer.ipynb                    <- shu dars")
print("  dataset/")
print("    train.bin                          <- {:,} token".format(n_train_tokens))
print("    val.bin                            <- {:,} token".format(n_val_tokens))
print("    meta.json                          <- dataset pasporti")
print("    tokenizer_report.md")


my_LLM/
  data/final/tinystories_final.jsonl   <- kirish
  tokenizer/
    tokenizer.json                     <- lug'at + merge rules
    train_tokenizer.py                 <- qayta yaratadigan skript
    tokenizer.ipynb                    <- shu dars
  dataset/
    train.bin                          <- 2,804,299 token
    val.bin                            <- 28,537 token
    meta.json                          <- dataset pasporti
    tokenizer_report.md


---

## 20. Xulosa

**Tokenizer haqida esda qoladigan beshta narsa:**

1. Subword - character (juda uzun) va word (OOV va cheksiz lug'at) o'rtasidagi
   yagona amaliy yechim.
2. BPE - takrorlanadigan argmax: eng ko'p uchragan juftlikni birlashtir, qoidani
   yozib qo'y. Natija - **tartiblangan qoidalar ro'yxati**, tartib muhim.
3. Byte-level -> `<unk>` fizik jihatdan mumkin emas, alifbo doim 256 ta.
4. `vocab_size` - embedding narxi va ketma-ketlik uzunligi o'rtasidagi savdo.
   Bizda 8000, va u `uint16` chegarasidan past bo'lishi shart.
5. Tokenizer - korpusning ko'zgusi. Uni faqat toza, yakuniy korpusda o'rgating.

**Dataset builder haqida:**

1. **Kodlashdan oldin** bo'ling - aks holda leakage.
2. `uint16` + `memmap` - RAM dan katta dataset ham ishlaydi.
3. Packing, padding emas - GPU ning 100% i haqiqiy tokenga ishlaydi.
4. `y[t] = x[t+1]` - butun language modeling shu bitta tenglikda.
5. `meta.json` bo'lmasa, `train.bin` bir haftadan keyin o'qib bo'lmaydigan
   raqamlar to'plami.

**Keyingi dars:** `train.bin` -> `nn.Embedding` -> self-attention -> Transformer bloki.